# Fruit data: exploration, correction and first models

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/abw/notebooks/lecture-3/fruit-data-exploration.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/courses/abw/notebooks/lecture-3/fruit-data-exploration.ipynb)

ABW lecture companion, originally developed by the ABW teaching team. This edition preserves the original sequence of data inspection, models and interpretation. Predict each result before running the cell.


## Prepare the libraries and original teaching data
Install missing libraries, then load the verified raw fruit table. Corrections happen in the lesson below, after inspecting the observations.


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'matplotlib': 'matplotlib', 'pandas': 'pandas', 'numpy': 'numpy', 'seaborn': 'seaborn', 'scipy': 'scipy', 'sklearn': 'scikit-learn'}
ensure_packages(required_packages)


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

# Prefer the checked-in file locally; Colab downloads the same frozen edition.
data_path = next((p for p in [Path("data/fruits.csv"), Path("fruits.csv")]
                 if p.is_file()), Path("fruits.csv"))
if not data_path.is_file():
    url = "https://raw.githubusercontent.com/gromicho/teaching/main/data/fruits.csv"
    with urlopen(url, timeout=45) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != "84235de12e6a8eb094423436b3cdd045c8b55288d773053c821d88db06443c22":
        raise ValueError("Dataset checksum mismatch; do not use an unverified copy.")
    data_path.write_bytes(payload)
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == "84235de12e6a8eb094423436b3cdd045c8b55288d773053c821d88db06443c22", "Unexpected local data version"


In [ ]:
import pandas as pd, seaborn as sns

# Get the data

In [ ]:
fruits = pd.read_csv(data_path, delimiter=';', decimal=',')
fruits

# Visualize the data

In [ ]:
sns.lmplot(x='Length', y='Width', data=fruits, hue='Name', fit_reg=False)

# 'Fix' the outliers
These two corrections represent known entry mistakes in this teaching dataset: swapped dimensions and a factor-of-ten unit error. An unusual observation alone does not establish a mistake. The extrema identify row labels (`idxmin`/`idxmax`), which remain valid with a non-default index. Apply these corrections once to the raw table.


In [ ]:
idx_min_length = fruits.Length.idxmin()
idx_max_width = fruits.Width.idxmax()

In [ ]:
fruits.at[ idx_max_width, 'Length' ],fruits.at[ idx_max_width, 'Width' ] = fruits.at[ idx_max_width, 'Width' ],fruits.at[ idx_max_width, 'Length' ]

In [ ]:
fruits.at[ idx_min_length, 'Length' ] = fruits.at[ idx_min_length, 'Length' ] * 10
fruits.at[ idx_min_length, 'Width' ] = fruits.at[ idx_min_length, 'Width' ] * 10

In [ ]:
sns.lmplot(x='Length', y='Width', data=fruits, hue='Name', fit_reg=False)

# Classification trees

In [ ]:
from sklearn import tree
import matplotlib.pyplot as plt
features = ['Length','Width']

In [ ]:
clf = tree.DecisionTreeClassifier(criterion='entropy', max_depth=2, random_state=0).fit(fruits[features], fruits.Name )
plt.figure(figsize=(12,12))  # set plot size (denoted in inches)
_ = tree.plot_tree(clf, filled=True, fontsize=10, feature_names=features, class_names=sorted( fruits.Name.unique() ) )

In [ ]:
fruits.Name.value_counts()

In [ ]:
fruits[ clf.predict( fruits[features] ) != fruits.Name ]

# Added to explain better the concept of entropy

In [ ]:
from math import log2

def Entropy( p ):
  return sum( [ -p*log2(p) if p > 0 else 0 for p in p ])

def EntropyNormalized( p ):
  return Entropy( p ) / log2(len(p))

from scipy.stats import entropy

In [ ]:
p = [ .5, .5 ]
entropy(p,base=2),EntropyNormalized(p),Entropy(p)

In [ ]:
counts = fruits.Name.value_counts().to_dict()
count = counts.values()
sum(count)

In [ ]:
p = [ c/sum(count) for c in count]

In [ ]:
entropy(p,base=2),EntropyNormalized(p),Entropy(p)

In [ ]:
node = [0,20,25,2]
p = [ c/sum(node) for c in node]

In [ ]:
entropy(p,base=2),EntropyNormalized(p),Entropy(p)

In [ ]:
4/40*Entropy( [4/4, 0/4] )+36/40*Entropy( [16/36, 20/36] )

In [ ]:
Entropy( [16/36, 20/36] )

# Clustering

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
kmeans = KMeans(n_clusters=2, n_init=10, random_state=0).fit(fruits[features])

In [ ]:
fruits['cluster'] = kmeans.labels_

In [ ]:
sns.lmplot(x='Length', y='Width', data=fruits, hue='cluster', fit_reg=False)

In [ ]:
sns.lmplot(x='Length', y='Width', data=fruits, hue='Name', fit_reg=False)

# Linear regression

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lin_reg = LinearRegression()  # Create linear regression object
x_train = fruits[fruits.cluster==0].Length.values.reshape(-1,1)
y_train = fruits[fruits.cluster==0].Width.values.reshape(-1,1)
lin_reg.fit( x_train, y_train )  # Train the model using the training sets

In [ ]:
model_line = lin_reg.predict(x_train)
plt.scatter(x_train, y_train, color='black')
plt.plot(x_train, model_line, color='blue', linewidth=3)
plt.xticks(())
plt.yticks(())
plt.show()

In [ ]:
# Intercept: the value for y when x=0 for the predicted line, \beta_0 in the formulas
lin_reg.intercept_

In [ ]:
# Coefficient: the slope of the predicted line, \beta_1 in the formulas
lin_reg.coef_

In [ ]:
import numpy as np
assert np.isclose(Entropy([0.5, 0.5]), 1)
assert np.isclose(Entropy([0, 1]), 0)
assert np.isclose(Entropy(p), entropy(p, base=2))


# Training fit and predictions for new observations

The tree's earlier list of mistakes uses the **same observations that trained the tree**. The regression line was also fitted and displayed on the same cluster. These results describe **training fit**: how well a model represents data it has already seen. They do not establish how accurately it predicts new observations.

We now demonstrate a different procedure: set aside a test set, fit new models using only the training set, and compare their training and test results. The test observations do not take part in fitting the models or their baselines.

**Predict:** will a model necessarily have worse results on the test set than on the training set? Would perfect training accuracy be sufficient evidence of useful predictions?

Because we have already explored this whole toy dataset, the exercise demonstrates the evaluation procedure; it is not a genuinely untouched final test of decisions made earlier in the notebook. In a real project, reserve the test set before exploring or choosing models. We fix the split and model settings below rather than trying alternatives until the test score looks good.

## Prepare a fresh copy and reserve a test set

Reload the raw table so rerunning this section cannot apply the earlier corrections twice. The two corrections below encode the **already documented entry errors** at records 19 and 20. They are fixed corrections, not rules estimated from the test data. An unfamiliar outlier in a new dataset would require investigation, not an automatic swap or multiplication.

Reserve 25% of observations for testing. Stratification keeps approximately the same fruit-class proportions in both sets. The fixed seed makes the example reproducible; it does not guarantee a representative test set. We use this same partition for both prediction tasks.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error, mean_squared_error
from IPython.display import display

evaluation_data = pd.read_csv(data_path, delimiter=';', decimal=',')
assert evaluation_data.loc[19, 'Name'] == 'Banana'
assert evaluation_data.loc[20, 'Name'] == 'Apple'
assert np.allclose(evaluation_data.loc[19, ['Length', 'Width']].astype(float), [2.5, 7.2])
assert np.allclose(evaluation_data.loc[20, ['Length', 'Width']].astype(float), [0.253, 0.277])
evaluation_data.loc[19, ['Length', 'Width']] = [7.2, 2.5]
evaluation_data.loc[20, ['Length', 'Width']] = [2.53, 2.77]

train_rows, test_rows = train_test_split(
    evaluation_data.index, test_size=0.25, random_state=7,
    stratify=evaluation_data['Name'],
)
training_fruits = evaluation_data.loc[train_rows].copy()
test_fruits = evaluation_data.loc[test_rows].copy()
assert set(train_rows).isdisjoint(test_rows)
assert set(train_rows) | set(test_rows) == set(evaluation_data.index)
split_counts = pd.DataFrame({
    'Training': training_fruits['Name'].value_counts(),
    'Test': test_fruits['Name'].value_counts(),
}).sort_index()
display(split_counts)
print(f'{len(training_fruits)} training observations; {len(test_fruits)} test observations.')

## Classification: predict the fruit name

Fit a new depth-2 decision tree using length and width from the training observations only. Compare it with a simple baseline that always predicts the most common **training** class. A useful model should justify its complexity relative to a relevant baseline.

Accuracy is the fraction of observations classified correctly. The table reports accuracy and the number of mistakes separately for the training and test sets. These sets have different sizes, so compare their accuracy values rather than just their mistake counts.

In [ ]:
classification_features = ['Length', 'Width']
X_class_train = training_fruits[classification_features]
y_class_train = training_fruits['Name']
X_class_test = test_fruits[classification_features]
y_class_test = test_fruits['Name']

heldout_tree = tree.DecisionTreeClassifier(criterion='entropy', max_depth=2, random_state=0)
majority_baseline = DummyClassifier(strategy='most_frequent')
heldout_tree.fit(X_class_train, y_class_train)
majority_baseline.fit(X_class_train, y_class_train)

classification_rows = []
for model_name, model in [('Depth-2 tree', heldout_tree), ('Most frequent class', majority_baseline)]:
    for split_name, X_part, y_part in [
        ('Training', X_class_train, y_class_train),
        ('Test', X_class_test, y_class_test),
    ]:
        predictions = model.predict(X_part)
        classification_rows.append({
            'Model': model_name, 'Data': split_name, 'Observations': len(y_part),
            'Mistakes': int(np.count_nonzero(predictions != y_part)),
            'Accuracy': accuracy_score(y_part, predictions),
        })
classification_results = pd.DataFrame(classification_rows)
display(classification_results.round(3))

The confusion matrix below uses **test observations only**. Rows show the actual fruit and columns show the prediction. Correct predictions are on the diagonal; the other entries show which fruits the tree confuses. The next table identifies the test mistakes without fitting the tree again.

In [ ]:
test_predictions = heldout_tree.predict(X_class_test)
class_order = sorted(evaluation_data['Name'].unique())
test_confusion = pd.DataFrame(
    confusion_matrix(y_class_test, test_predictions, labels=class_order),
    index=pd.Index(class_order, name='Actual'),
    columns=pd.Index(class_order, name='Predicted'),
)
display(test_confusion)
test_mistakes = test_fruits.assign(Predicted=test_predictions)
display(test_mistakes.loc[test_mistakes['Name'] != test_mistakes['Predicted']])

## Regression: predict width from length

For this task, assume we know a new fruit's **length** and want to predict its **width**. Fit one line across all training fruits and compare it with a baseline that always predicts the mean width in the training set.

This task differs from the earlier regression within cluster 0. That cluster was constructed using **both length and width**. Using a test fruit's width to choose its cluster would use the quantity we are trying to predict. Fitting K-means on training data alone would not solve that problem if assigning new fruits still required their unknown width. Here we therefore avoid clustering and use length alone. Compare the training and test errors of this new model, not its errors with those of the earlier subgroup model.

For residuals $e_i=y_i-\hat y_i$, we report

$$\mathrm{MAE}=\frac{1}{m}\sum_{i=1}^{m}|e_i|,\qquad
\mathrm{RMSE}=\sqrt{\frac{1}{m}\sum_{i=1}^{m}e_i^2}.$$

Here $m$ is the number of observations being evaluated. Both measures have the same units as width; lower is better. RMSE gives larger errors more weight. Regression has no classification-style percentage accuracy unless we first define what counts as an acceptable numerical error.

In [ ]:
X_reg_train = training_fruits[['Length']]
y_reg_train = training_fruits['Width']
X_reg_test = test_fruits[['Length']]
y_reg_test = test_fruits['Width']

heldout_regression = LinearRegression().fit(X_reg_train, y_reg_train)
mean_baseline = DummyRegressor(strategy='mean').fit(X_reg_train, y_reg_train)
regression_rows = []
for model_name, model in [('Linear regression', heldout_regression), ('Training mean', mean_baseline)]:
    for split_name, X_part, y_part in [
        ('Training', X_reg_train, y_reg_train),
        ('Test', X_reg_test, y_reg_test),
    ]:
        predictions = model.predict(X_part)
        regression_rows.append({
            'Model': model_name, 'Data': split_name, 'Observations': len(y_part),
            'MAE': mean_absolute_error(y_part, predictions),
            'RMSE': np.sqrt(mean_squared_error(y_part, predictions)),
        })
regression_results = pd.DataFrame(regression_rows)
display(regression_results.round(3))
print(f'Training-fitted line: predicted width = {heldout_regression.intercept_:.3f}'
      f' + ({heldout_regression.coef_[0]:.3f}) * length')

In [ ]:
length_grid = pd.DataFrame({
    'Length': np.linspace(evaluation_data['Length'].min(), evaluation_data['Length'].max(), 100),
})
fig, ax = plt.subplots(figsize=(8, 5), layout='constrained')
ax.scatter(training_fruits['Length'], training_fruits['Width'],
           color='0.6', alpha=0.65, label='Training observations')
ax.scatter(test_fruits['Length'], test_fruits['Width'],
           color='#D68124', marker='x', s=55, label='Test observations')
ax.plot(length_grid['Length'], heldout_regression.predict(length_grid),
        color='#2878B5', label='Line fitted on training data')
ax.axhline(float(mean_baseline.constant_.item()), color='0.3', linestyle='--',
           label='Mean of training widths')
ax.set(xlabel='Length', ylabel='Width', title='Width predictions from length alone')
ax.legend()
plt.show()

## What can we conclude?

- **Read the test results and the baselines together.** Does the tree beat the most frequent class? Does the line beat the training mean? A small training error alone does not answer either question about new observations.
- **A test score need not be worse than a training score.** A small test set can contain easier or harder observations by chance. The gap is not guaranteed to have one direction.
- **One line may not describe all fruits well.** The regression plot can reveal model limitations. Evaluating a weak model honestly does not make it a good model.
- **Do not tune on these test results.** If you change tree depth, features, preprocessing or model choice after inspecting them, this set has become part of model development. Use validation data or cross-validation within the training set for those choices, and reserve a separate final test set.
- **Fit learned preprocessing on training data only.** This includes imputation, scaling, feature selection and clustering. Apply the fitted transformations to new inputs without using their targets. Known entry corrections are different from learning a cleaning rule from the whole dataset.
- **The split must match the intended use.** This exercise treats rows as separate observations from the same population. Repeated measurements of the same fruit should stay together; predicting future observations can require a chronological split. Neither a random split nor a high score guarantees performance on a different population.

With just 23 test fruits, one additional classification mistake changes accuracy by about 4.3 percentage points. This is an illustrative estimate on a small teaching dataset, not a certificate of real-world performance.

**Explain in your own words:** why are the original training mistakes still useful? What did holding out observations change? Why would using width to choose a regression cluster be inappropriate when width is the prediction target?

Further reading: scikit-learn's [training/test splits and cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html), [data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage), and [evaluation metrics](https://scikit-learn.org/stable/modules/model_evaluation.html).